In [1]:
!pip install -q langchain-community langchain-google-genai langchain-core faiss-cpu scikit-learn pandas numpy tqdm
!pip install --upgrade langchain
!pip install --upgrade langchain-text-splitters
!pip install --upgrade tqdm
!pip install langchain_huggingface



   ------ --------------------------------- 1/6 [langgraph-sdk]
   ------ --------------------------------- 1/6 [langgraph-sdk]
   ------ --------------------------------- 1/6 [langgraph-sdk]
   ------------- -------------------------- 2/6 [langgraph-checkpoint]
   ------------- -------------------------- 2/6 [langgraph-checkpoint]
   -------------------- ------------------- 3/6 [langgraph-prebuilt]
   -------------------------- ------------- 4/6 [langgraph]
   -------------------------- ------------- 4/6 [langgraph]
   -------------------------- ------------- 4/6 [langgraph]
   -------------------------- ------------- 4/6 [langgraph]
   -------------------------- ------------- 4/6 [langgraph]
   -------------------------- ------------- 4/6 [langgraph]
   -------------------------- ------------- 4/6 [langgraph]
   --------------------------------- ------ 5/6 [langchain]
   --------------------------------- ------ 5/6 [langchain]
   --------------------------------- ------ 5/6 [langcha

In [2]:
!pip install faiss-cpu
from langchain_community.vectorstores import FAISS

In [ ]:
import pandas as pd
import os
import json

# === Configuration - Local paths ===
DOCUMENTS_PATH = "Dataset/export_1"
RES_PATH = "RES.xlsx"
RESULTS_DIR = "report/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# === Load RES dataset ===
df = pd.read_excel(RES_PATH)
# Only keep relevant columns
df = df[["Câu Hỏi", "Trả lời"]].dropna()
df.to_csv("res.csv", index=False)

print(f"Total Q&A pairs: {len(df)}")
print(f"Documents path: {DOCUMENTS_PATH}")
print(f"Total documents: {len([f for f in os.listdir(DOCUMENTS_PATH) if f.endswith('.txt')])}")
print(f"\nSample question: {df.iloc[0]['Câu Hỏi'][:100]}...")
print(f"Sample answer: {df.iloc[0]['Trả lời'][:100]}...")

## Phase 2: Data Exploration & Understanding
### 2.1 Legal Documents Corpus Analysis

In [ ]:
# ============================================================
# Phase 2.1: Legal Documents Corpus Analysis
# ============================================================
import os
import json
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

DOCUMENTS_PATH = "Dataset/export_1"
RESULTS_DIR = "report/results"

# --- Load all documents and compute stats ---
doc_lengths = []
doc_names = []
all_text = []

for filename in sorted(os.listdir(DOCUMENTS_PATH)):
    if filename.endswith(".txt"):
        filepath = os.path.join(DOCUMENTS_PATH, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
        doc_lengths.append(len(content))
        doc_names.append(filename)
        all_text.append(content.lower())

doc_lengths = np.array(doc_lengths)

corpus_stats = {
    "total_documents": len(doc_lengths),
    "total_characters": int(doc_lengths.sum()),
    "length_min": int(doc_lengths.min()),
    "length_max": int(doc_lengths.max()),
    "length_mean": round(float(doc_lengths.mean()), 1),
    "length_median": round(float(np.median(doc_lengths)), 1),
    "length_std": round(float(doc_lengths.std()), 1),
    "length_q25": round(float(np.percentile(doc_lengths, 25)), 1),
    "length_q75": round(float(np.percentile(doc_lengths, 75)), 1),
}

# --- Classify document types by filename prefix ---
type_patterns = {
    "Thông tư (TT)": r"^\d*-?TT",
    "Nghị định (NĐ)": r"^\d*-?NĐ|NĐ-CP",
    "Quyết định (QĐ)": r"^\d*-?QĐ|QĐ-",
    "Nghị quyết (NQ)": r"^\d*-?NQ",
    "Chỉ thị (CT)": r"^\d*-?CT",
    "Luật": r"^luật|_luật",
    "Thông báo (TB)": r"^\d*-?TB",
    "Công văn (CV)": r"^\d*-?CV",
    "Chính phủ (CP)": r"^\d*-?CP",
    "UBND": r"UBND",
}

doc_types = Counter()
for name in doc_names:
    classified = False
    for label, pattern in type_patterns.items():
        if re.search(pattern, name, re.IGNORECASE):
            doc_types[label] += 1
            classified = True
            break
    if not classified:
        doc_types["Khác"] += 1

corpus_stats["doc_types"] = dict(doc_types.most_common())

# --- Save stats ---
with open(os.path.join(RESULTS_DIR, "corpus_stats.json"), "w", encoding="utf-8") as f:
    json.dump(corpus_stats, f, ensure_ascii=False, indent=2)

print("=== CORPUS STATISTICS ===")
for k, v in corpus_stats.items():
    if k != "doc_types":
        print(f"  {k}: {v}")
print(f"\n  Document types:")
for dtype, count in doc_types.most_common():
    print(f"    {dtype}: {count}")

# --- Plot 1: Document length distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(doc_lengths, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Document Length (characters)')
axes[0].set_ylabel('Number of Documents')
axes[0].set_title('Distribution of Document Lengths')
axes[0].axvline(doc_lengths.mean(), color='red', linestyle='--', label=f'Mean: {doc_lengths.mean():.0f}')
axes[0].axvline(np.median(doc_lengths), color='orange', linestyle='--', label=f'Median: {np.median(doc_lengths):.0f}')
axes[0].legend()

# Log scale for better visibility
axes[1].hist(doc_lengths, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Document Length (characters)')
axes[1].set_ylabel('Number of Documents (log scale)')
axes[1].set_title('Distribution of Document Lengths (Log Scale)')
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "doc_length_distribution.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR}/doc_length_distribution.png")

# --- Plot 2: Document type distribution ---
types_sorted = doc_types.most_common()
labels = [t[0] for t in types_sorted]
counts = [t[1] for t in types_sorted]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(labels[::-1], counts[::-1], color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('Number of Documents')
ax.set_title('Distribution of Legal Document Types')
for bar, count in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2, str(count), va='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "doc_type_distribution.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR}/doc_type_distribution.png")

### 2.2 RES Dataset (Q&A Pairs) Analysis

In [ ]:
# ============================================================
# Phase 2.2: RES Dataset (Q&A Pairs) Analysis
# ============================================================
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

RESULTS_DIR = "report/results"
df = pd.read_csv("res.csv")

# --- Compute stats ---
q_lengths_char = df["Câu Hỏi"].str.len()
a_lengths_char = df["Trả lời"].str.len()
q_lengths_word = df["Câu Hỏi"].str.split().str.len()
a_lengths_word = df["Trả lời"].str.split().str.len()

res_stats = {
    "total_qa_pairs": len(df),
    "question": {
        "length_char_min": int(q_lengths_char.min()),
        "length_char_max": int(q_lengths_char.max()),
        "length_char_mean": round(float(q_lengths_char.mean()), 1),
        "length_char_median": round(float(q_lengths_char.median()), 1),
        "length_word_min": int(q_lengths_word.min()),
        "length_word_max": int(q_lengths_word.max()),
        "length_word_mean": round(float(q_lengths_word.mean()), 1),
    },
    "answer": {
        "length_char_min": int(a_lengths_char.min()),
        "length_char_max": int(a_lengths_char.max()),
        "length_char_mean": round(float(a_lengths_char.mean()), 1),
        "length_char_median": round(float(a_lengths_char.median()), 1),
        "length_word_min": int(a_lengths_word.min()),
        "length_word_max": int(a_lengths_word.max()),
        "length_word_mean": round(float(a_lengths_word.mean()), 1),
    }
}

with open(f"{RESULTS_DIR}/res_stats.json", "w", encoding="utf-8") as f:
    json.dump(res_stats, f, ensure_ascii=False, indent=2)

print("=== RES DATASET STATISTICS ===")
print(f"  Total Q&A pairs: {res_stats['total_qa_pairs']}")
print(f"\n  Questions:")
for k, v in res_stats['question'].items():
    print(f"    {k}: {v}")
print(f"\n  Answers:")
for k, v in res_stats['answer'].items():
    print(f"    {k}: {v}")

# --- Plot: Q&A length distributions ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(q_lengths_char, bins=50, color='#2196F3', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Question Length (characters)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Question Length Distribution (chars)')
axes[0, 0].axvline(q_lengths_char.mean(), color='red', linestyle='--', label=f'Mean: {q_lengths_char.mean():.0f}')
axes[0, 0].legend()

axes[0, 1].hist(q_lengths_word, bins=50, color='#2196F3', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Question Length (words)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Question Length Distribution (words)')
axes[0, 1].axvline(q_lengths_word.mean(), color='red', linestyle='--', label=f'Mean: {q_lengths_word.mean():.0f}')
axes[0, 1].legend()

axes[1, 0].hist(a_lengths_char, bins=50, color='#4CAF50', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Answer Length (characters)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Answer Length Distribution (chars)')
axes[1, 0].axvline(a_lengths_char.mean(), color='red', linestyle='--', label=f'Mean: {a_lengths_char.mean():.0f}')
axes[1, 0].legend()

axes[1, 1].hist(a_lengths_word, bins=50, color='#4CAF50', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Answer Length (words)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Answer Length Distribution (words)')
axes[1, 1].axvline(a_lengths_word.mean(), color='red', linestyle='--', label=f'Mean: {a_lengths_word.mean():.0f}')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/qa_length_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR}/qa_length_distribution.png")

# --- Save sample Q&A pairs (diverse selection) ---
sample_indices = np.linspace(0, len(df)-1, 10, dtype=int)
sample_df = df.iloc[sample_indices][["Câu Hỏi", "Trả lời"]].copy()
sample_df.columns = ["Question", "Answer"]
sample_df["Answer_Truncated"] = sample_df["Answer"].str[:200] + "..."
sample_df.to_csv(f"{RESULTS_DIR}/sample_qa_pairs.csv", index=False, encoding="utf-8-sig")
print(f"\nSaved: {RESULTS_DIR}/sample_qa_pairs.csv")

# Display samples
for i, row in sample_df.iterrows():
    print(f"\n--- Q{i} ---")
    print(f"  Q: {row['Question'][:120]}...")
    print(f"  A: {row['Answer_Truncated'][:120]}...")

### 2.3 Chunking Analysis & Top Keywords

In [ ]:
# ============================================================
# Phase 2.3: Chunking Analysis & Top Keywords
# ============================================================
import os
import json
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from langchain_text_splitters import RecursiveCharacterTextSplitter

DOCUMENTS_PATH = "Dataset/export_1"
RESULTS_DIR = "report/results"

# --- Load documents (reuse if already loaded) ---
print("Loading documents for chunking analysis...")
raw_contents = []
for filename in sorted(os.listdir(DOCUMENTS_PATH)):
    if filename.endswith(".txt"):
        with open(os.path.join(DOCUMENTS_PATH, filename), 'r', encoding='utf-8') as f:
            raw_contents.append(f.read())

# --- Baseline chunking config ---
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=50,
    length_function=len
)

all_chunks = []
chunks_per_doc = []
for content in raw_contents:
    chunks = text_splitter.split_text(content)
    all_chunks.extend(chunks)
    chunks_per_doc.append(len(chunks))

chunk_lengths = np.array([len(c) for c in all_chunks])
chunks_per_doc = np.array(chunks_per_doc)

chunk_stats = {
    "chunking_config": {
        "chunk_size": 2000,
        "chunk_overlap": 50,
        "method": "RecursiveCharacterTextSplitter"
    },
    "total_chunks": len(all_chunks),
    "total_documents": len(raw_contents),
    "chunks_per_doc_mean": round(float(chunks_per_doc.mean()), 2),
    "chunks_per_doc_median": round(float(np.median(chunks_per_doc)), 1),
    "chunks_per_doc_max": int(chunks_per_doc.max()),
    "chunk_length_min": int(chunk_lengths.min()),
    "chunk_length_max": int(chunk_lengths.max()),
    "chunk_length_mean": round(float(chunk_lengths.mean()), 1),
    "chunk_length_median": round(float(np.median(chunk_lengths)), 1),
}

with open(f"{RESULTS_DIR}/chunk_stats.json", "w", encoding="utf-8") as f:
    json.dump(chunk_stats, f, ensure_ascii=False, indent=2)

print("=== CHUNK STATISTICS (Baseline Config) ===")
for k, v in chunk_stats.items():
    if k != "chunking_config":
        print(f"  {k}: {v}")

# --- Plot: Chunk length distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(chunk_lengths, bins=50, color='#FF9800', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Chunk Length (characters)')
axes[0].set_ylabel('Number of Chunks')
axes[0].set_title(f'Chunk Length Distribution (n={len(all_chunks):,})')
axes[0].axvline(chunk_lengths.mean(), color='red', linestyle='--', label=f'Mean: {chunk_lengths.mean():.0f}')
axes[0].legend()

axes[1].hist(chunks_per_doc, bins=range(0, int(chunks_per_doc.max())+2), color='#FF9800', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Chunks per Document')
axes[1].set_ylabel('Number of Documents')
axes[1].set_title('Chunks per Document Distribution')
axes[1].axvline(chunks_per_doc.mean(), color='red', linestyle='--', label=f'Mean: {chunks_per_doc.mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/chunk_length_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR}/chunk_length_distribution.png")

# --- Top Keywords (from all documents) ---
print("\nComputing top keywords...")

# Vietnamese stopwords (common ones)
vn_stopwords = set("và của là trong có được cho này với các không một những để theo về từ đã khi tại như trên đến hoặc nhưng cũng do nếu sẽ vì còn hay người năm đó bị phải ra đi qua tuy rằng".split())

word_counter = Counter()
for content in raw_contents:
    text = content.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    words = [w for w in text.split() if len(w) > 1 and w not in vn_stopwords and not w.isdigit()]
    word_counter.update(words)

top_words = word_counter.most_common(30)

fig, ax = plt.subplots(figsize=(12, 8))
words = [w[0] for w in top_words[::-1]]
freqs = [w[1] for w in top_words[::-1]]
ax.barh(words, freqs, color='#9C27B0', edgecolor='black', alpha=0.7)
ax.set_xlabel('Frequency')
ax.set_title('Top 30 Keywords in Legal Document Corpus')
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/top_keywords.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR}/top_keywords.png")

# Save top keywords
with open(f"{RESULTS_DIR}/top_keywords.json", "w", encoding="utf-8") as f:
    json.dump(top_words, f, ensure_ascii=False, indent=2)

## Phase 1: Baseline Setup & Evaluation
### 1.1 Build FAISS Vectorstore (skip if already saved)

In [ ]:
import os
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# === Configuration ===
DOCUMENTS_PATH = "Dataset/export_1"  # Local path
EMBEDDING_MODEL = "keepitreal/vietnamese-sbert"
BATCH_SIZE = 1024
OUTPUT_DIR = "faiss_db"

# === Load text documents ===
def load_text_documents(path):
    documents = []
    for filename in os.listdir(path):
        if filename.endswith(".txt"):
            file_path = os.path.join(path, filename)
            loader = TextLoader(file_path, encoding='utf-8')
            raw_docs = loader.load()
            for doc in raw_docs:
                content = doc.page_content.replace("\n", " ").lower().strip()
                doc.page_content = content
                documents.append(doc)
    return documents

print("Loading text documents...")
raw_documents = load_text_documents(DOCUMENTS_PATH)
print(f"Total documents found: {len(raw_documents)}")

# === Split documents into smaller chunks ===
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=50,
    length_function=len
)

texts = []
for doc in tqdm(raw_documents, desc="Splitting documents into chunks"):
    chunks = text_splitter.split_documents([doc])
    texts.extend(chunks)

print(f"Total chunks created: {len(texts)}")

# === Initialize embeddings ===
print(f"Loading embedding model: {EMBEDDING_MODEL}")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cuda"}
)

# === Build FAISS vectorstore ===
vectorstore = None

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding and adding to FAISS"):
    batch = texts[i:i + BATCH_SIZE]
    batch_embeddings = embeddings.embed_documents([t.page_content for t in batch])

    if vectorstore is None:
        vectorstore = FAISS.from_texts(
            [t.page_content for t in batch],
            embeddings
        )
    else:
        vectorstore.add_texts(
            [t.page_content for t in batch],
            embedding=batch_embeddings
        )

# === Save FAISS index ===
os.makedirs(OUTPUT_DIR, exist_ok=True)
vectorstore.save_local(OUTPUT_DIR)
print(f"FAISS vectorstore successfully saved to '{OUTPUT_DIR}'")

### 1.2 Load FAISS & Verify

In [ ]:
import numpy as np
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# === Configuration ===
EMBEDDING_MODEL = "keepitreal/vietnamese-sbert"
VECTORSTORE_PATH = "faiss_db"

# === Load FAISS vectorstore ===
print("Loading FAISS vectorstore...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cuda"}  # change to "cpu" if no GPU
)

vectorstore = FAISS.load_local(
    VECTORSTORE_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)
print("FAISS vectorstore loaded successfully.")

# === Display a sample record ===
keys = list(vectorstore.docstore._dict.keys())
print(f"Total records in vectorstore: {len(keys)}")

first_key = keys[0]
record = vectorstore.docstore._dict[first_key]
print(f"\n--- Sample Record ---")
print(f"Content (truncated): {record.page_content[:200]}...")

# === Vector info ===
vector = vectorstore.index.reconstruct(0)
print(f"\nVector dimension: {len(vector)}")
print(f"First 5 values: {vector[:5]}")

### 1.3 Baseline Evaluation (≥100 Q&A pairs)

In [ ]:
import os
import re
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import Counter
from langchain_huggingface import HuggingFacePipeline
from sklearn.metrics.pairwise import cosine_similarity

RESULTS_DIR = "report/results"
NUM_EVAL = 100  # Minimum 100, increase if time allows

# ==================== LOAD DATA ====================
df = pd.read_csv("res.csv")
df = df.head(NUM_EVAL)
print(f"Total number of questions for evaluation: {len(df)}")

# ==================== LOAD LLM ====================
llm = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen1.5-1.8B",
    task="text-generation",
    device=0,  # GPU 0
    model_kwargs={"temperature": 0, "max_length": 1024}
)

# ==================== HELPER FUNCTIONS ====================
def normalize_text(text: str) -> str:
    text = str(text).lower().replace("\n", " ").strip()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

def rag_query(question: str, k: int = 5) -> str:
    docs = vectorstore.similarity_search(question, k=k)
    context_texts = []
    for doc in docs:
        if hasattr(doc, "page_content"):
            context_texts.append(doc.page_content)
        elif isinstance(doc, dict) and "page_content" in doc:
            context_texts.append(doc["page_content"])
        else:
            context_texts.append(str(doc))
    context = "\n".join(context_texts)

    prompt = f"""Bạn là một trợ lý thông minh. Trả lời câu hỏi dựa trên ngữ cảnh sau.
Nếu không đủ thông tin, hãy nói rõ điều đó, không được tự bịa đặt.

Ngữ cảnh:
{context}

Câu hỏi: {question}
Trả lời ngắn gọn và chính xác nhất có thể:
"""
    output = llm.generate([prompt], max_new_tokens=256)
    generated_text = output.generations[0][0].text
    return generated_text

def cosine_similarity_score(pred, true, emb_model):
    if not pred or not true: return 0.0
    return float(cosine_similarity([emb_model.embed_query(pred)], [emb_model.embed_query(true)])[0][0])

def jaccard_similarity(pred, true):
    p, t = set(normalize_text(pred).split()), set(normalize_text(true).split())
    return len(p & t) / len(p | t) if p and t else 0.0

def token_overlap_score(pred, true):
    p, t = normalize_text(pred).split(), normalize_text(true).split()
    return sum((Counter(p) & Counter(t)).values()) / len(t) if t else 0.0

def bleu_score_simple(pred, true):
    p, t = normalize_text(pred).split(), normalize_text(true).split()
    if not p or not t: return 0.0
    overlap = sum((Counter(p) & Counter(t)).values())
    precision = overlap / len(p)
    bp = 1.0 if len(p) >= len(t) else np.exp(1 - len(t)/len(p))
    return bp * precision

def rouge_l_score(pred, true):
    p, t = normalize_text(pred).split(), normalize_text(true).split()
    if not p or not t: return 0.0
    m, n = len(p), len(t)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i][j] = dp[i-1][j-1] + 1 if p[i-1] == t[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    prec, rec = lcs/m, lcs/n
    return 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0

# ==================== EVALUATION ====================
results = []
responses = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating Baseline RAG"):
    question = row["Câu Hỏi"]
    ground_truth = str(row["Trả lời"]).strip()

    try:
        response = rag_query(question)
    except Exception as e:
        print(f"Error at question {idx}: {e}")
        response = ""

    responses.append(response)
    metrics = {
        "Question": question[:100],
        "Cosine_Similarity": cosine_similarity_score(response, ground_truth, embeddings),
        "Jaccard_Similarity": jaccard_similarity(response, ground_truth),
        "Token_Overlap": token_overlap_score(response, ground_truth),
        "BLEU_Score": bleu_score_simple(response, ground_truth),
        "ROUGE_L": rouge_l_score(response, ground_truth)
    }
    results.append(metrics)

results_df = pd.DataFrame(results)
metric_cols = ["Cosine_Similarity", "Jaccard_Similarity", "Token_Overlap", "BLEU_Score", "ROUGE_L"]
mean_metrics = results_df[metric_cols].mean().to_dict()

# ==================== SAVE RESULTS ====================
# 1. Per-question metrics
results_df.to_csv(f"{RESULTS_DIR}/baseline_metrics.csv", index=False, encoding="utf-8-sig")

# 2. Summary
summary_lines = [
    "=== BASELINE RAG EVALUATION SUMMARY ===",
    f"Number of Q&A pairs evaluated: {NUM_EVAL}",
    f"Embedding Model: keepitreal/vietnamese-sbert",
    f"LLM: Qwen/Qwen1.5-1.8B",
    f"Chunk Size: 2000, Overlap: 50",
    f"Top-k: 5",
    "",
]
for k, v in mean_metrics.items():
    summary_lines.append(f"{k}: {v:.4f}")

with open(f"{RESULTS_DIR}/baseline_summary.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

# 3. Baseline config
config_md = """# Baseline Configuration

| Component | Detail |
|---|---|
| **Chunking** | 2000 chars/chunk, overlap 50 chars |
| **Embedding Model** | `keepitreal/vietnamese-sbert` (768 dim) |
| **Vector DB** | FAISS (local) |
| **Top-k retrieval** | k = 5 |
| **LLM** | `Qwen/Qwen1.5-1.8B` (1.8B params) |
| **Framework** | LangChain |
| **Eval Q&A pairs** | """ + str(NUM_EVAL) + """ |
"""
with open(f"{RESULTS_DIR}/baseline_config.md", "w", encoding="utf-8") as f:
    f.write(config_md)

# ==================== DISPLAY & PLOT ====================
print("\n=== BASELINE EVALUATION SUMMARY ===")
for k, v in mean_metrics.items():
    print(f"  {k}: {v:.4f}")

# Plot metrics distribution
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, col in enumerate(metric_cols):
    axes[i].hist(results_df[col], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(col.replace('_', ' '))
    axes[i].axvline(results_df[col].mean(), color='red', linestyle='--', label=f'Mean: {results_df[col].mean():.3f}')
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel('Score')
    axes[i].set_ylabel('Count')

plt.suptitle(f'Baseline Metrics Distribution (n={NUM_EVAL})', fontsize=14)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/baseline_metrics_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSaved to {RESULTS_DIR}/:")
print("  - baseline_metrics.csv")
print("  - baseline_summary.txt")
print("  - baseline_config.md")
print("  - baseline_metrics_distribution.png")